# ReCAP study — false-alarm calibration (causal eval)

Upload this notebook from local path:
`D:\\quant\\ReCAP\\notebooks\\study_run_colab.ipynb`

What this run produces:
- fit once at default h=2.5 sigma
- calibrate thresholds on train for 0.5 / 1 / 2 alarms per year
- causal test eval for default + calibrated variants
- synthetic-shift detector diagnostic

Uses **micro** profile for Colab RAM. Protocol-correct pilot for the paper draft.

In [ ]:
REPO_URL = "https://github.com/laraib-b/ReCAP-study.git"
REPO_DIR = "/content/ReCAP"
BRANCH = "study-sunday"

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo = Path(REPO_DIR)
if not repo.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
%pip install -q "numpy>=1.24,<2.5" "pandas==2.2.3" "torch" "gymnasium>=0.29" "stable-baselines3>=2.0" "yfinance>=0.2" "requests>=2.31"
import os
os.environ["YF_DISABLE_CURL_CFFI"] = "1"
print("deps ready")

## Run study (micro)

Expect several minutes to tens of minutes on CPU.
When finished, download `outputs/study/*.json`.

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
!python study/run_study.py --profile micro --dataset DOW30 --device {device} --seed 0 --disable-local-cache --no-adapt-control

In [ ]:
from pathlib import Path
import json

files = sorted(Path("outputs/study").glob("study_*.json"))
print("files:", [p.name for p in files])
if not files:
    raise SystemExit("No study JSON found")
latest = files[-1]
!python study/aggregate.py {latest}
payload = json.loads(latest.read_text())
print("variants:", list(payload["results"]))

## After success

1. Download latest `outputs/study/study_*.json` and the `.md` table.
2. Message Cursor that the study run finished.
3. Optional: rerun with `--seed 1` then `--seed 2`.